# When a model makes two things

The [5-minute tour](../docs/showcase.md) leaves one thread hanging. Its cement
works asks for steam, nothing produces the exact concept it asks for, and a
generic `HeatSupply` answers the generalised demand. That supplier is
monofunctional — it makes heat and nothing else — which kept the tour on one
argument at a time.

Real heat suppliers are often not. A combined heat and power plant makes heat
*and* electricity from one lot of fuel, and how the fuel's burden splits between
the two is a choice. `trailrunner` will not make that choice for you.

This notebook picks the thread up: same cement demand, same chain, but the steam
comes from a CHP.

`GasCHP` below is written here rather than shipped, and its efficiencies and
prices are invented. What is being demonstrated is the rule's effect on the
answer and the fact that the run records which rule it used — not this
particular plant.

Nothing here touches the network.

In [1]:
import sys
from pathlib import Path

EXAMPLES = Path.cwd() if (Path.cwd() / "showcase_models.py").exists() else Path.cwd() / "examples"
sys.path.insert(0, str(EXAMPLES))

from showcase_models import MODELS
from trailrunner import (
    AttributionSettings, Demand, Exchange, Flow, Glossary, LocationHierarchy,
    Model, Orchestrator, ParameterSet, Property, Result, Settings,
)
from trailrunner.assessment import Method, assess
from trailrunner.models import cement
from trailrunner.models.cement import CEMENT, CO2_FOSSIL, ELECTRICITY, NATURAL_GAS
from trailrunner.resolution import (
    GeneralisingProvider, ModelProvider, PystLabels, PystTaxonomy, ResolutionChain,
)

HIERARCHY = LocationHierarchy({"CH": "RER", "FR": "RER", "RER": "GLO"})
VOCAB = PystLabels(EXAMPLES / "pyst_labels.json", client=None)

DEMAND = Demand(
    flow=Flow(iri=CEMENT, location="CH", time=2030), amount=1000.0, unit="kg"
)
print(DEMAND.amount, DEMAND.unit, VOCAB.label(CEMENT))

1000.0 kg Portland cement, aluminous cement, slag cement and similar hydraulic cements, except in the form of clinkers


## A model that co-produces

`GasCHP` declares two products. It also declares which allocation rules it can
honour — and `none` is not among them, because there is no honest way to answer
a demand for its heat without saying what to do with its electricity.

Each product carries a `price` property. Economic allocation needs one; a rule
that partitions by something else would need that something else instead, and
the model states what it has rather than guessing what the study will want.

In [2]:
STEAM_AND_HOT_WATER = "https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_1730"


class GasCHP(Model):
    """Gas-fired combined heat and power. Illustrative efficiencies and prices."""

    produces = [STEAM_AND_HOT_WATER]
    supports = frozenset({"economic", "substitution"})  # it co-produces

    heat_efficiency = 0.50
    electrical_efficiency = 0.35
    co2_per_mj_fuel = 0.056  # the same factor examples/gas_power_params.parquet carries
    heat_price = 0.02        # EUR/MJ
    electricity_price = 0.10  # EUR/kWh

    def apply(self, demand):
        here = {"location": demand.flow.location, "time": demand.flow.time}
        fuel = demand.amount / self.heat_efficiency
        power = fuel * self.electrical_efficiency / 3.6
        return Result(
            production=[
                Exchange(flow=demand.flow, amount=demand.amount, unit=demand.unit,
                         properties=(Property("price", demand.amount * self.heat_price, "EUR"),)),
                Exchange(flow=Flow(iri=ELECTRICITY, **here), amount=power, unit="kWh",
                         properties=(Property("price", power * self.electricity_price, "EUR"),)),
            ],
            technosphere=[Demand(flow=Flow(iri=NATURAL_GAS, **here), amount=fuel, unit="MJ")],
            biosphere=[Exchange(flow=Flow(iri=CO2_FOSSIL, **here),
                                amount=fuel * self.co2_per_mj_fuel, unit="kg")],
            provenance={"fuel_mj": fuel},
        )

In [3]:
cement_params = ParameterSet.from_parquet(
    EXAMPLES / "cement_params.parquet", hierarchy=HIERARCHY
)

MODELS_PLUS = [*MODELS, GasCHP()]
tier1 = ModelProvider(Glossary(MODELS_PLUS))
taxonomy = PystTaxonomy(EXAMPLES / "pyst_cache.json", client=None)  # client=None: no network
CHAIN = ResolutionChain([tier1, GeneralisingProvider(tier1, hierarchy=HIERARCHY, taxonomy=taxonomy)])


def walk(allocation):
    settings = Settings(attribution=AttributionSettings(allocation=allocation))
    return Orchestrator(CHAIN, settings=settings).calculate(DEMAND)

## The refusal

The default rule is `none`, which means *this study has not chosen*. Running
under it reaches the CHP, finds a second product, and stops.

The refusal happens in the `Runner`, between applying the model and validating
what came back, so the model neither makes the choice nor sees it.

In [4]:
from trailrunner import UnallocatedCoProduction

try:
    walk("none")
except UnallocatedCoProduction as refusal:
    print("allocation='none' ->", refusal)

allocation='none' -> GasCHP returned co-products (https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_17100) but the run's allocation rule is 'none'; model it monofunctionally or choose a rule


## Choose a rule, and the run carries it

Two rules, the same model, the same 1000 kg of cement.

`economic` partitions the CHP's burden between heat and electricity by their
revenue. `substitution` instead credits the electricity: the co-product goes
back on the queue as a *negative* demand, is answered by someone other than the
CHP, and brings its own cutoffs, counted separately.

In [5]:
GWP100 = Method(
    rows=[
        {"flow_iri": "https://vocab.sentier.dev/flows/co2-fossil", "flow_unit": "kg",
         "location": "GLO", "cf": 1.0},
        {"flow_iri": "https://vocab.sentier.dev/flows/ch4-fossil", "flow_unit": "kg",
         "location": "GLO", "cf": 29.8},
        {"flow_iri": "https://vocab.sentier.dev/flows/n2o", "flow_unit": "kg",
         "location": "GLO", "cf": 273.0},
    ],
    unit="kg CO2-eq",
    name="IPCC AR6 GWP100",
    hierarchy=HIERARCHY,
)

runs = {rule: walk(rule) for rule in ("economic", "substitution")}

for rule, run in runs.items():
    score = assess(run, GWP100).score
    print(f"{rule:>13}: {score:>9.1f} kg CO2-eq   ({run.summary().splitlines()[1]})")

credited = runs["substitution"]
chp_node = [node for node in credited.nodes if node.demand.flow.iri == cement.STEAM][0]
print()
print("what the CHP node recorded under substitution:")
for key, value in credited.attribution[chp_node.id].items():
    print(f"  {key}: {value}")

     economic:     556.1 kg CO2-eq   (6 unresolved (generalisation_exhausted: 6))
 substitution:     574.4 kg CO2-eq   (9 unresolved (generalisation_exhausted: 9, of which 3 on a credit branch))

what the CHP node recorded under substitution:
  allocation: substitution
  share: 1.0
  substituted: ['https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_17100']


Two numbers for one question, and the gap between them is set by `GasCHP`'s
invented prices, since economic allocation partitions by revenue. That is the
point rather than a weakness of the example: the rule moves the answer, the
choice belongs to the study, and the report records which one ran.

---

- The tour this picks up from: [`examples/showcase.ipynb`](showcase.ipynb)
- The rules themselves: [Attribution](../docs/content/attribution.md)